In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

## Load Data

In [ ]:
# If running on colab you can download the data using gdown. Uncomment the below code. 
# import gdown
# !gdown 1hVRVEC7siG7ROyVDaEZrKjHUaWLJYaY1 #x_train_img.npz
# !gdown 1bLJrl5Dv7hi8K7-0LQtpnsF8FOPqlzIs #x_test_img.npz

In [2]:
img_file = "x_train_img.npz"
metadata_file = "x_train.csv"
y_file = "y_train.csv"

In [3]:
X_dev = pd.read_csv(metadata_file)
y_dev = pd.read_csv(y_file)
X_dev

,patient_id,img_id,age,region,itch,grew,hurt,changed,bleed,elevation
0,PAT_960,PAT_960_0001,62,NOSE,False,True,False,False,False,True
1,PAT_601,PAT_601_0002,70,FACE,True,False,True,False,True,True
2,PAT_816,PAT_816_0003,71,HAND,True,False,False,False,False,False
3,PAT_083,PAT_083_0005,36,CHEST,False,True,False,False,False,True
4,PAT_897,PAT_897_0008,71,NOSE,True,False,True,False,True,True
...,...,...,...,...,...,...,...,...,...,...
1509,PAT_467,PAT_467_1888,54,THIGH,False,True,True,False,False,True
1510,PAT_1120,PAT_1120_1889,55,BACK,False,False,False,False,False,True
1511,PAT_635,PAT_635_1890,68,ABDOMEN,True,True,False,False,False,True
1512,PAT_190,PAT_190_1892,66,FACE,True,True,False,False,True,True


In [ ]:
# Function and code to load images
def load_img_data(file_path):
    with np.load(file_path, allow_pickle=True) as data:
        img = data['images']
        ids = data['image_ids']
    print(f"Successfully loaded {img.shape[0]} images.")  
    return img, ids

imgs, img_ids = load_img_data(img_file)
print(img_ids.shape, imgs.shape)

# Make sure the IDs match and are in the same order
assert np.all(img_ids == X_dev['img_id']), "Image IDs in metadata and image data do not match"

ValueError: Object arrays cannot be loaded when allow_pickle=False

In [ ]:
# Plot the class balance
sns.countplot(y_dev["fine_label"])

In [ ]:
# Show the first 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))
for i in range(5):
    axes[i].imshow(imgs[i])
    axes[i].axis('off')
    axes[i].set_title(f"Sample {i} ({y_dev.iloc[i]["fine_label"]})")

plt.show()

## Encode Categorical Features

We'll convert all categorical features to one-hot encodings, and all binary features to 1/0. 
We can auto-detect the binary features by looking for "True" and "False" (after manually looking through the data to confirm that's how these features are encoded.) Note you'll also want to apply this code to x_test for your final predictions. 

In [ ]:
# Get all obvious true/false cols that aren't already true/false
for col in X_dev.select_dtypes(include=['object']):
    if X_dev[col].dropna().astype(str).str.contains('True|False').any():
        print(f"Automatically converting {col} to binary...")
        unmapped = X_dev[~X_dev[col].isin(['True','False'])][col].unique()
        print("The following values are being mapped to NaN in this column: ", unmapped)
        X_dev[col] = X_dev[col].map({'True': 1, 'False': 0, True: 1, False: 0})

# Encode region with a one-hot encoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
region_dev_encoded = ohe.fit_transform(X_dev[['region']])

region_cols = ohe.get_feature_names_out(['region'])
region_dev_df = pd.DataFrame(region_dev_encoded, columns=region_cols, index=X_dev.index)
X_dev = pd.concat([X_dev.drop(columns=['region']), region_dev_df], axis=1)
print("New encoded region columns:", list(region_cols))

### Make age groups feature

In [ ]:
age_groups = pd.cut(
    X_dev['age'], 
    bins=[0, 30, 60, 100], 
    labels=['Young (<30)', 'Adult (30-60)', 'Senior (60+)']
)

## Your Code

You might want to revisit the above preprocessing steps for your Problem 2 model. You also might want to put some of the above code into functions to make it easier to apply to different datasets. 